In [ ]:
import os
import yaml
import torch
import lightning as pl
from pprint import pprint

from src.data.utils_data import MRIDataModule


from src.model.base_3d import UNetAlzheimer3D as BaseUnet
from src.model.advance_3d import UNetAlzheimer3D as AdvanceUnet
from src.model.advance_att_3d import UNetAlzheimer3D as AttAdvanceUnet

# Read configuration file
with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)
print("== Configuration File ==")
pprint(cfg)

DEV = torch.device(cfg["project"]["device"])
print("device used: ", DEV)
ROOT = os.getcwd()
print("root path: ", ROOT)
CKPT_PATH = os.path.join(ROOT, cfg["training"]["ckpt_path"])
print("ckpt path: ", CKPT_PATH)

In [ ]:
TEST_CKPT = os.path.join(CKPT_PATH, f"more_advance/g6hxmxat/model-epoch={5}.ckpt")
print(TEST_CKPT)

In [ ]:
# test_dataloader = data_manager.test_dataloader()
RELOAD_DATA = True
data_path = os.path.join(ROOT, cfg["paths"]["data"])
data_manager = MRIDataModule(dataset=torch.load(os.path.join(data_path, "complete.db"), weights_only=False),
                                 test_set=torch.load(os.path.join(data_path, "complete.db"), weights_only=False),
                                 saved_db_folder=os.path.join(ROOT, "data_split/complete"),  # os.path.join(data_path, f"complete"),
                                 batch_size=1, reload_data=RELOAD_DATA, drop_last=False)

# Model Instatiation
u_net = AttAdvanceUnet.load_from_checkpoint(checkpoint_path=TEST_CKPT,
                                            class_weights={"train_weights": data_manager.training_class_weigths}).to(device=DEV)
u_net.eval()

trainer = pl.Trainer(accelerator=cfg["project"]["device"],        # gpu, cpu, mps
                     num_sanity_val_steps=0,
                     devices=1, max_epochs=1, log_every_n_steps=1)

# Start Testing Procedure
trainer.test(u_net, dataloaders=data_manager.test_dataloader())
print("Test Complete")

# Gradient Cam

## Extract One Batch

In [ ]:
# Create an iterator from the DataLoader
test_iter = iter(data_manager.test_dataloader())

# Get the first batch
test_batch = next(test_iter)

pprint(test_batch)
print(len(test_batch))

## Model Computation

In [ ]:
u_net = UNetAlzheimer3D.load_from_checkpoint(checkpoint_path=prova_checkpoint,
                                             class_weights={"train_weights": data_manager.training_class_weigths}).to(device="cpu")
u_net.eval()

prova_unet_out, prova_unet_features_3, prova_unet_features_2, prova_unet_features_1 = u_net(test_batch["Image"].to(device="cpu"),
                                                                                            sex=test_batch["Sex"].to(device="cpu"),
                                                                                            age=test_batch["Age"].to(device="cpu"))

In [ ]:
prova_labels = test_batch["Disease"]
if len(prova_labels.shape) < 2: prova_labels = prova_labels.unsqueeze(dim=0)
prova_labels = torch.argmax(prova_labels, dim=-1)

# Disease Classification
prova_output_pred = prova_unet_out.squeeze()
if len(prova_output_pred.shape) < 2: prova_output_pred = prova_output_pred.unsqueeze(dim=0)
prova_output_pred = F.softmax(prova_output_pred, dim=1) # Disease Features probability distribution
if len(prova_output_pred.shape) < 2: output_pred = prova_output_pred.unsqueeze(dim=0)

print("Loss: ", u_net.loss_function(prova_output_pred, prova_labels).detach().cpu().item());
print("Accuracy: ", u_net.disease_accuracy(prova_output_pred, prova_labels).detach().cpu().item());

## Explainability: Gradient Cam

In [ ]:
from src.utils.plots import plot_features_in_the_batch
from src.utils.utils_grad import grad_cam

In [ ]:
# def overlay_heatmap(heatmaps, batch_tensor:torch.Tensor, samples:int=2, verbose:bool=False, color_map:str="gray"):
#     for batch_sample, heatmap in zip(batch_tensor, heatmaps):
#         batch_i = batch_sample[0].detach().cpu().numpy()
#         print("Batch Example Shape: ", batch_i.shape)
#         for i in range(samples):
#             if verbose ==True: print(batch_sample[i])
#             print("Heatmap Shape: ", heatmap[i].shape)
#             # Create a figure and a 1x3 grid of subplots arranged horizontally
#             fig, axes = plt.subplots(1, 3, figsize=(20, 3))
            
#             axial = batch_i[:, :, batch_i.shape[2]//2]
#             coronal = np.rot90(batch_i[:, batch_i.shape[1]//2, :], k=-1) # counter-clockwise
#             sagittal = np.rot90(batch_i[batch_i.shape[0]//2, :, :], k=-1) # counter-clockwise
            
#             # Resize the heatmap to match the original image size
#             heatmap_axial = heatmap[i][:, :, heatmap[i].shape[2]//2]
#             heatmap_coronal = np.rot90(heatmap[i][:, heatmap[i].shape[1]//2, :])
#             heatmap_sagittal = np.rot90(heatmap[i][heatmap[i].shape[0]//2, :, :])
            
#             # Resize img
#             axial = cv2.resize(axial, (heatmap_axial.shape[1], heatmap_axial.shape[0]))#; axial = np.expand_dims(axial, axis=-1)
#             coronal = cv2.resize(coronal, (heatmap_coronal.shape[1], heatmap_coronal.shape[0]))#; coronal = np.expand_dims(coronal, axis=-1)
#             sagittal = cv2.resize(sagittal, (heatmap_sagittal.shape[1], heatmap_sagittal.shape[0]))#; sagittal = np.expand_dims(sagittal, axis=-1)
#             print("axial shape: ", axial.shape)
#             print("coronal shape: ", axial.shape)
#             print("sagittal shape: ", axial.shape)
            
#             # Overlay CAM on the original image
#             # heatmap_axial = np.uint8(255 * heatmap_axial)
#             # heatmap_coronal = np.uint8(255 * heatmap_coronal)
#             # heatmap_sagittal = np.uint8(255 * heatmap_sagittal)

#             # Convert the normalized heatmap to a colormap (e.g., 'viridis')
#             heatmap_axial = plt.cm.jet(heatmap_axial)
#             heatmap_coronal = plt.cm.jet(heatmap_coronal)
#             heatmap_sagittal = plt.cm.jet(heatmap_sagittal)
#             print("heatmap_axial shape: ", heatmap_axial.shape)
#             print("heatmap_coronal shape: ", heatmap_coronal.shape)
#             print("heatmap_sagittal shape: ", heatmap_sagittal.shape)

#             # Overlay the heatmap on the black and white image
#             alpha = 0.5  # Set the transparency level for the heatmap overlay
#             axial = alpha * heatmap_axial[:, :, :3] + (1 - alpha) * np.expand_dims(2*axial, axis=2)
#             coronal = alpha * heatmap_coronal[:, :, :3] + (1 - alpha) * np.expand_dims(2*coronal, axis=2)
#             sagittal = alpha * heatmap_sagittal[:, :, :3] + (1 - alpha) * np.expand_dims(2*sagittal, axis=2)


#             # Convert single-channel axial image to three channels
#             # axial = np.dstack([axial] * 3)
#             # sagittal = np.dstack([sagittal] * 3)
#             # coronal = np.dstack([coronal] * 3)
            
#             # # Blend the heatmap with the original image
#             # axial = axial * 1.0 + heatmap_axial * 2.0
#             # coronal = coronal * 1.0 + heatmap_coronal * 2.0
#             # sagittal = sagittal * 1.0 + heatmap_sagittal * 2.0
#             # print("axial shape: ", axial.shape)
#             # print("coronal shape: ", axial.shape)
#             # print("sagittal shape: ", axial.shape)
            
#             # Plot the data on each subplot
#             im_0 = axes[0].imshow(axial, cmap=color_map)
#             axes[0].set_title('Assiale (Sopra-Sotto)')
#             im_1 = axes[1].imshow(coronal, cmap=color_map)
#             axes[1].set_title('Coronale (Fronte-Retro)')
#             im_2 = axes[2].imshow(sagittal, cmap=color_map)
#             axes[2].set_title('Sagittal (Laterale)')

#             # Add colorbars
#             cbar0 = fig.colorbar(im_0, ax=axes[0])
#             cbar0.set_label('Intensity')

#             cbar1 = fig.colorbar(im_1, ax=axes[1])
#             cbar1.set_label('Intensity')

#             cbar2 = fig.colorbar(im_2, ax=axes[2])
#             cbar2.set_label('Intensity')

#             # Adjust spacing between subplots
#             plt.tight_layout()
#             # Show the plot
#             plt.show()
#         print("#######")

# overlay_heatmap(heatmaps=heatmap.unsqueeze(dim=0), batch_tensor=test_batch["Image"], samples=8, color_map="jet")

In [ ]:
heatmap = grad_cam(model_out=prova_output_pred[0][0], model_features=prova_unet_features_1, model=u_net)
print("Heatmap shape: ", heatmap.shape)
print("Disease: ", test_batch["Disease"])
plot_features_in_the_batch(test_batch["Image"], samples=test_batch["Image"].shape[0], color_map="gray")
plot_features_in_the_batch(heatmap.unsqueeze(dim=0), samples=8, color_map="jet")

## Fetaures in the batch

In [ ]:
plot_features_in_the_batch(test_batch["Image"], samples=test_batch["Image"].shape[0], color_map="gray")
print(prova_unet_features_1.shape)
plot_features_in_the_batch(prova_unet_features_1, samples=prova_unet_features_1.shape[1], color_map="jet")